# ArtDiffuse-RL: Kaggle Training

Train a small diffusion model on Midjourney text-to-image data with 2x T4 GPUs.

In [ ]:
# Clone the repository
!git clone https://github.com/10xdev4u-alt/artdiffuse-rl.git /kaggle/working/artdiffuse-rl
import sys
sys.path.insert(0, '/kaggle/working/artdiffuse-rl/src')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Install dependencies
!pip install -q pyyaml tqdm pillow requests

In [ ]:
# Load config
from train import load_config, build_model, train
from dataset import MidjourneyDataset
from torch.utils.data import DataLoader

config = load_config('/kaggle/working/artdiffuse-rl/configs/base.yaml')

# Adjust for Kaggle
config['training']['batch_size'] = 256
config['training']['max_steps'] = 30000
config['training']['save_every'] = 5000
config['data']['num_workers'] = 4

print('Config:')
for k, v in config.items():
    print(f'  {k}: {v}')

In [ ]:
# Create dataset
dataset = MidjourneyDataset(
    data_dir='/kaggle/input/midjourney-texttoimage',
    image_size=64,
    max_samples=50000
)

dataloader = DataLoader(
    dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['data']['num_workers'],
    pin_memory=True
)

print(f'Dataset: {len(dataset)} samples')
print(f'Batches: {len(dataloader)}')

In [ ]:
# Build model
device = torch.device('cuda')
model = build_model(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB')

In [ ]:
# Train
print('Starting training...')
model = train(
    model=model,
    dataloader=dataloader,
    config=config['training'],
    device=device,
    output_dir='/kaggle/working/models'
)

In [ ]:
# Generate samples
from evaluate import save_samples

model.eval()
with torch.no_grad():
    samples = model.sample((16, 3, 64, 64), device)

save_samples(samples, '/kaggle/working/results')
print('Training complete!')